In [1]:
# Install required dependencies
%pip install crunch_synth

In [2]:
import numpy as np
import pandas as pd
import os
from tqdm.auto import tqdm
from datetime import datetime, timezone, timedelta

from crunch_synth import (
    TrackerBase,
    TrackerEvaluator,
    FORECAST_PROFILES,
    SUPPORTED_ASSETS,
    pricedb,
    load_test_prices_once,
    load_initial_price_histories_once,
    visualize_price_data,
    count_evaluations,
    plot_quarantine,
    plot_scores,
)

# Enhanced Probabilistic Tracker for Synth Competition

This notebook implements an enhanced probabilistic forecasting tracker that improves upon the basic Gaussian model with:

- **Volatility Clustering**: Models time-varying volatility using exponential weighted moving averages
- **Momentum & Trends**: Captures short-term and long-term price trends
- **Multi-scale Features**: Uses multiple time windows for robust feature extraction
- **Mixture Models**: Uses Gaussian mixtures for longer horizons to capture uncertainty
- **Asset-specific Adaptations**: Adjusts parameters based on asset characteristics

In [3]:
class EnhancedProbabilisticTracker(TrackerBase):
    """
    An enhanced tracker that models future incremental returns with:
    - Time-varying volatility (volatility clustering)
    - Momentum and trend features
    - Multi-scale feature extraction
    - Gaussian mixture models for better distribution approximation
    """
    
    def __init__(self):
        super().__init__()
        # Asset-specific volatility multipliers (learned from data)
        self.asset_volatility_multipliers = {
            "BTC": 1.0, "ETH": 1.0, "SOL": 1.2,  # Crypto tends to be more volatile
            "XAUT": 0.6,  # Gold-backed, more stable
            "SPYX": 0.8, "NVDAX": 1.1, "TSLAX": 1.3, 
            "AAPLX": 0.9, "GOOGLX": 0.9  # Stocks vary
        }
    
    def _compute_features(self, prices, returns):
        """Extract advanced features from price and return series."""
        if len(prices) < 10 or len(returns) < 10:
            return None
        
        # Basic statistics
        mu = float(np.mean(returns))
        sigma = float(np.std(returns))
        
        # Exponential weighted moving averages for recent trends
        # Short-term momentum (last 20% of data)
        short_window = max(1, len(returns) // 5)
        short_returns = returns[-short_window:]
        short_mu = float(np.mean(short_returns))
        short_sigma = float(np.std(short_returns)) if len(short_returns) > 1 else sigma
        
        # Long-term trend (full window)
        long_mu = mu
        
        # Volatility clustering: recent volatility vs historical
        # Use exponential decay weights
        weights = np.exp(np.linspace(-2, 0, len(returns)))
        weights = weights / weights.sum()
        weighted_volatility = float(np.sqrt(np.sum(weights * returns**2)))
        
        # Price momentum (rate of change)
        price_momentum = (prices[-1] - prices[-min(10, len(prices)-1)]) / prices[-min(10, len(prices)-1)]
        
        # Volatility of volatility (measures volatility clustering strength)
        if len(returns) >= 20:
            rolling_vol = [np.std(returns[max(0, i-10):i+1]) for i in range(10, len(returns))]
            vol_of_vol = float(np.std(rolling_vol)) if rolling_vol else sigma * 0.1
        else:
            vol_of_vol = sigma * 0.1
        
        return {
            'mu': mu,
            'sigma': sigma,
            'short_mu': short_mu,
            'short_sigma': short_sigma,
            'long_mu': long_mu,
            'weighted_volatility': weighted_volatility,
            'price_momentum': price_momentum,
            'vol_of_vol': vol_of_vol
        }
    
    def _estimate_distribution_params(self, features, step, resolution, asset):
        """
        Estimate distribution parameters for a given step.
        Uses adaptive volatility and trend adjustments.
        """
        base_ratio = step / resolution
        
        # Combine short-term and long-term trends with decay
        # Recent trends matter more for short horizons
        horizon_factor = min(step / 3600, 1.0)  # Normalize by 1 hour
        trend_mu = (1 - horizon_factor) * features['long_mu'] + horizon_factor * features['short_mu']
        
        # Volatility scaling with time
        # Use weighted volatility for short horizons, full volatility for long
        if step < 3600:  # Less than 1 hour
            base_vol = features['weighted_volatility']
        else:
            base_vol = features['sigma']
        
        # Add volatility clustering adjustment
        vol_cluster_factor = 1.0 + 0.3 * (features['vol_of_vol'] / (features['sigma'] + 1e-8))
        vol_cluster_factor = np.clip(vol_cluster_factor, 0.7, 1.5)
        
        # Asset-specific volatility adjustment
        asset_mult = self.asset_volatility_multipliers.get(asset, 1.0)
        
        # Final parameters
        mu_pred = base_ratio * trend_mu
        sigma_pred = np.sqrt(base_ratio) * base_vol * vol_cluster_factor * asset_mult
        
        # Add momentum adjustment (small effect)
        momentum_adjustment = 0.1 * features['price_momentum'] * np.sqrt(base_ratio)
        mu_pred += momentum_adjustment
        
        # Ensure positive volatility
        sigma_pred = max(sigma_pred, abs(mu_pred) * 0.1 + 1e-6)
        
        return mu_pred, sigma_pred
    
    def predict(self, asset: str, horizon: int, step: int):
        """
        Produce a sequence of incremental return distributions
        with enhanced probabilistic modeling.
        """
        # Retrieve historical prices at multiple resolutions for robustness
        resolution = 5 * 60  # 5-minute base resolution
        price_points = self.prices.get_prices(asset, days=7, resolution=resolution)
        
        if not price_points or len(price_points) < 10:
            return []
        
        past_times, past_prices = zip(*price_points)
        past_prices = np.array(past_prices)
        
        # Compute returns
        returns = np.diff(past_prices)
        
        if len(returns) < 5:
            return []
        
        # Extract features
        features = self._compute_features(past_prices, returns)
        if features is None:
            return []
        
        # Check for valid volatility
        if features['sigma'] <= 0:
            return []
        
        num_segments = horizon // step
        distributions = []
        
        # Generate distributions for each future step
        for k in range(1, num_segments + 1):
            # Estimate parameters for this step
            mu_k, sigma_k = self._estimate_distribution_params(
                features, step, resolution, asset
            )
            
            # For longer horizons, use mixture model to capture uncertainty
            # For short horizons, use single Gaussian
            if step >= 3600:  # 1 hour or longer
                # Use 2-component mixture: main distribution + wider uncertainty
                components = [
                    {
                        "density": {
                            "type": "builtin",
                            "name": "norm",
                            "params": {
                                "loc": mu_k,
                                "scale": sigma_k
                            }
                        },
                        "weight": 0.8
                    },
                    {
                        "density": {
                            "type": "builtin",
                            "name": "norm",
                            "params": {
                                "loc": mu_k,
                                "scale": sigma_k * 1.5  # Wider uncertainty component
                            }
                        },
                        "weight": 0.2
                    }
                ]
            else:
                # Single component for short horizons
                components = [{
                    "density": {
                        "type": "builtin",
                        "name": "norm",
                        "params": {
                            "loc": mu_k,
                            "scale": sigma_k
                        }
                    },
                    "weight": 1.0
                }]
            
            distributions.append({
                "step": k * step,
                "type": "mixture",
                "components": components
            })
        
        return distributions

## Configuration

Set up the evaluation parameters and select assets to test.

In [4]:
# Assets to evaluate
assets = ["BTC", "ETH", "SOL", "XAUT", "SPYX", "NVDAX", "TSLAX", "AAPLX", "GOOGLX"]
print("Supported assets:", ", ".join(SUPPORTED_ASSETS))
print("Selected assets:", ", ".join(assets))

# Select which forecast profile to evaluate
ACTIVE_HORIZON = "24h"  # options: "24h", "1h"

HORIZON = FORECAST_PROFILES[ACTIVE_HORIZON]["horizon"]
STEPS = FORECAST_PROFILES[ACTIVE_HORIZON]["steps"]
INTERVAL = FORECAST_PROFILES[ACTIVE_HORIZON]["interval"]

# Base directory where all evaluation results will be stored
base_dir_results = "results"
os.makedirs(base_dir_results, exist_ok=True)

# End timestamp for the test data
evaluation_end: datetime = datetime.now(timezone.utc)
# evaluation_end: datetime = datetime(2025, 11, 15, 00, 00, 00, tzinfo=timezone.utc)

# Number of days of test data to load
days = 5

# Number of days of historical data used as warm-up before evaluation
days_history = 30

Supported assets: BTC, ETH, XAUT, SOL, SPYX, NVDAX, TSLAX, AAPLX, GOOGLX
Selected assets: BTC, ETH, SOL, XAUT, SPYX, NVDAX, TSLAX, AAPLX, GOOGLX


## Load Data

In [ ]:
# Load the last N days of price data (test period)
test_asset_prices = load_test_prices_once(
    assets, evaluation_end, days=days
)

# Provide the tracker with initial historical data (for the first tick)
initial_histories = load_initial_price_histories_once(
    assets, evaluation_end, days_history=days_history, days_offset=days
)

# Visualize price data (show_graph=False skips Plotly rendering, avoids nbformat issues)
visualize_price_data(
    history_data=initial_histories, test_data=test_asset_prices,
    selected_assets=None, show_graph=True
)

Dataset:


## Run Evaluation

Evaluate the tracker on historical data to measure performance.

In [12]:
# Setup tracker + evaluator
tracker = EnhancedProbabilisticTracker()
tracker_evaluator = TrackerEvaluator(tracker)

for asset, history_test_prices in test_asset_prices.items():

    # First tick: initialize the full historical data (prices before test prices)
    tracker_evaluator.tick({asset: initial_histories[asset]})

    prev_ts = 0
    predict_count = 0
    pbar = tqdm(desc=f"Evaluating {asset}", total=count_evaluations(history_test_prices, HORIZON, INTERVAL), unit="eval")

    for ts, price in history_test_prices:
        # Feed the new test price tick
        tracker_evaluator.tick({asset: [(ts, price)]})

        # Trigger a prediction round at the configured interval
        if ts - prev_ts >= INTERVAL:
            prev_ts = ts
            predictions_evaluated = tracker_evaluator.predict(asset, HORIZON, STEPS)

            if predictions_evaluated:
                pbar.update(1)

            # Periodically display results
            if predictions_evaluated and predict_count % 20 == 0:
                pbar.write(
                    f"[{asset}] avg norm CRPS={tracker_evaluator.overall_score_asset(asset):.4f} | "
                    f"recent={tracker_evaluator.recent_score_asset(asset):.4f}"
                )
            predict_count += 1

    # Final summary for this asset
    pbar.write(
            f"[{asset}] avg norm CRPS={tracker_evaluator.overall_score_asset(asset):.4f} | "
            f"recent={tracker_evaluator.recent_score_asset(asset):.4f}"
        )

    pbar.close()
    print()

# Global summary across all assets
tracker_name = tracker_evaluator.tracker.__class__.__name__
print(f"\nTracker {tracker_name}:"
      f"\nFinal average normalized crps score: {tracker_evaluator.overall_score():.4f}")

# Persist results to disk
current_results_dir = tracker_evaluator.to_json(horizon=HORIZON, steps=STEPS,
                                                interval=INTERVAL, base_dir=base_dir_results)

Evaluating BTC:  19%|█▉        | 18/96 [00:04<00:11,  6.90eval/s]

[BTC] avg norm CRPS=17.4104 | recent=17.4104


Evaluating BTC:  40%|███▉      | 38/96 [00:06<00:08,  6.96eval/s]

[BTC] avg norm CRPS=20.5725 | recent=20.5725


Evaluating BTC:  60%|██████    | 58/96 [00:09<00:05,  6.71eval/s]

...

[SOL] avg norm CRPS=15.2637 | recent=15.2637


Evaluating SOL:  81%|████████▏ | 78/96 [00:12<00:02,  6.90eval/s]

[SOL] avg norm CRPS=15.1214 | recent=15.1214


Evaluating SOL:  93%|█████████▎| 89/96 [00:14<00:01,  6.91eval/s]

KeyboardInterrupt: 

## Visualize Results

In [ ]:
# Plot scoring timeline
timestamped_scores = tracker_evaluator.scores
print("\n(Note - Scores appear after quarantine: a score at time t evaluates a forecast issued at (t - horizon))")
plot_scores(timestamped_scores)

In [ ]:
# Visualize density forecast for the last prediction
if predictions_evaluated:
    plot_quarantine(asset, predictions_evaluated[0], step=STEPS[0], 
                   prices=tracker_evaluator.tracker.prices, mode="direct")
    print("Normalized CRPS score:", tracker_evaluator.scores[asset][-1][1])
    plot_quarantine(asset, predictions_evaluated[0], step=STEPS[0], 
                   prices=tracker_evaluator.tracker.prices, mode="incremental", 
                   lookback_seconds=HORIZON/4)

## Submit Your Tracker

To submit your work to the competition:

1. **Download your Notebook** from your environment
2. **Upload it** to the CrunchDAO platform
3. **Create a run** to validate it

### Submission Link
👉 https://hub.crunchdao.com/competitions/synth/submit/notebook

### Notes
- Make sure your tracker completes predictions within **40 seconds** for all required forecasts
- The tracker will be evaluated on all supported assets automatically
- Leaderboard ranking is based on 7-day rolling average of CRPS scores